# 4. Petrophysics and volumetrics

        We calculate porosity, hydrocarbon pore volume, and volumetric original oil in place.
        Fractions are stored as fractions—not percentages—and invalid inputs are rejected.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [2]:
from petroleum_python.petrophysics import hydrocarbon_pore_volume, oil_in_place_stb, porosity
phi = porosity(pore_volume=22.5, bulk_volume=100.0)
hcpv_m3 = hydrocarbon_pore_volume(1_000_000, phi, water_saturation=0.28)
print(f"Porosity: {phi:.1%}")
print(f"Hydrocarbon pore volume: {hcpv_m3:,.0f} m³")


Porosity: 22.5%
Hydrocarbon pore volume: 162,000 m³


In [3]:
wells = pd.read_csv(ROOT / "data" / "synthetic_wells.csv")
wells["net_pay_ft"] = wells["net_pay_m"] * 3.28084
wells["drainage_area_acres"] = 40.0
wells["bo_rb_stb"] = 1.22
wells["ooip_mstb"] = wells.apply(
    lambda row: oil_in_place_stb(
        row["drainage_area_acres"], row["net_pay_ft"], row["porosity_fraction"],
        row["water_saturation_fraction"], row["bo_rb_stb"]
    ) / 1_000,
    axis=1,
)
wells[["well_id", "field", "ooip_mstb"]].sort_values("ooip_mstb", ascending=False).head()


,well_id,field,ooip_mstb
1,W-02,North,3291.041720
20,W-21,North,3220.492401
25,W-26,South,2828.541922
24,W-25,North,2765.644768
18,W-19,Central,2668.852332


## Interpretation

        Volumetric OOIP is an in-place estimate, not recoverable reserves. Drainage area and
        formation volume factor are simplified assumptions here and must be justified in real work.